In [12]:
import os
import pandas as pd
import tkinter as tk
from tkinter import ttk, filedialog, messagebox, scrolledtext
from datetime import datetime
import threading
import glob
import warnings
warnings.filterwarnings('ignore')

class ExcelSummaryGenerator:
    def __init__(self):
        self.root = tk.Tk()
        self.root.title("Excel 요약 결과 추가 도구")
        self.root.geometry("700x500")
        
        self.is_running = False
        self.stop_requested = False
        
        self.setup_ui()
    
    def setup_ui(self):
        # 메인 프레임
        main_frame = tk.Frame(self.root)
        main_frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)
        
        # 제목
        tk.Label(main_frame, text="Excel 요약 결과 추가 도구", 
                font=('Arial', 14, 'bold'), fg='#2E86C1').pack(pady=(0, 10))
        
        # 파일 설정
        config_frame = tk.LabelFrame(main_frame, text="파일 설정", font=('Arial', 10, 'bold'))
        config_frame.pack(fill=tk.X, pady=(0, 10))
        
        # 입력 폴더
        input_frame = tk.Frame(config_frame)
        input_frame.pack(fill=tk.X, padx=5, pady=3)
        tk.Label(input_frame, text="처리된 Excel 폴더:", width=15).pack(side=tk.LEFT)
        self.input_folder_var = tk.StringVar(value="./excel_files_결과")
        tk.Entry(input_frame, textvariable=self.input_folder_var, font=('Arial', 9)).pack(side=tk.LEFT, padx=5, fill=tk.X, expand=True)
        tk.Button(input_frame, text="찾기", command=self.browse_input_folder, width=8).pack(side=tk.RIGHT)
        
        # 출력 폴더
        output_frame = tk.Frame(config_frame)
        output_frame.pack(fill=tk.X, padx=5, pady=3)
        tk.Label(output_frame, text="결과 저장 폴더:", width=15).pack(side=tk.LEFT)
        self.output_folder_var = tk.StringVar(value="./excel_files_요약완성")
        tk.Entry(output_frame, textvariable=self.output_folder_var, font=('Arial', 9)).pack(side=tk.LEFT, padx=5, fill=tk.X, expand=True)
        tk.Button(output_frame, text="찾기", command=self.browse_output_folder, width=8).pack(side=tk.RIGHT)
        
        # 컨트롤 버튼
        button_frame = tk.Frame(main_frame)
        button_frame.pack(fill=tk.X, pady=10)
        
        self.start_button = tk.Button(button_frame, text="시작", command=self.start_processing,
                                     font=('Arial', 11, 'bold'), bg='#28a745', fg='white', width=10)
        self.start_button.pack(side=tk.LEFT, padx=5)
        
        self.stop_button = tk.Button(button_frame, text="중지", command=self.stop_processing,
                                    font=('Arial', 11, 'bold'), bg='#dc3545', fg='white', 
                                    state=tk.DISABLED, width=10)
        self.stop_button.pack(side=tk.LEFT, padx=5)
        
        # 상태 표시
        self.status_var = tk.StringVar(value="준비")
        tk.Label(button_frame, textvariable=self.status_var, font=('Arial', 10), fg='blue').pack(side=tk.RIGHT, padx=10)
        
        # 진행률
        self.progress = ttk.Progressbar(main_frame, mode='indeterminate')
        self.progress.pack(fill=tk.X, pady=5)
        
        # 로그
        tk.Label(main_frame, text="실행 로그", font=('Arial', 10, 'bold')).pack(anchor=tk.W)
        self.log_text = scrolledtext.ScrolledText(main_frame, wrap=tk.WORD, font=('Consolas', 8), 
                                                 height=15, state=tk.DISABLED)
        self.log_text.pack(fill=tk.BOTH, expand=True, pady=5)
        
        self.add_log("Excel 요약 결과 추가 도구가 준비되었습니다.")
    
    def browse_input_folder(self):
        folder = filedialog.askdirectory(title="처리된 Excel 파일이 있는 폴더를 선택하세요")
        if folder:
            self.input_folder_var.set(folder)
            self.add_log(f"입력 폴더: {folder}")
    
    def browse_output_folder(self):
        folder = filedialog.askdirectory(title="결과를 저장할 폴더를 선택하세요")
        if folder:
            self.output_folder_var.set(folder)
            self.add_log(f"출력 폴더: {folder}")
    
    def add_log(self, message):
        try:
            timestamp = datetime.now().strftime("%H:%M:%S")
            self.log_text.config(state=tk.NORMAL)
            self.log_text.insert(tk.END, f"[{timestamp}] {message}\n")
            self.log_text.config(state=tk.DISABLED)
            self.log_text.see(tk.END)
            self.root.update_idletasks()
        except:
            print(f"[{timestamp}] {message}")
    
    def start_processing(self):
        if self.is_running:
            return
        
        input_folder = self.input_folder_var.get()
        if not os.path.exists(input_folder):
            messagebox.showerror("오류", f"입력 폴더가 존재하지 않습니다:\n{input_folder}")
            return
        
        self.is_running = True
        self.stop_requested = False
        self.start_button.config(state=tk.DISABLED)
        self.stop_button.config(state=tk.NORMAL)
        self.progress.start()
        self.status_var.set("처리 중...")
        
        threading.Thread(target=self.run_processing, daemon=True).start()
    
    def stop_processing(self):
        self.stop_requested = True
        self.add_log("중지 요청됨")
    
    def run_processing(self):
        try:
            input_folder = self.input_folder_var.get()
            output_folder = self.output_folder_var.get()
            
            # 출력 폴더 생성
            os.makedirs(output_folder, exist_ok=True)
            self.add_log(f"출력 폴더 준비: {output_folder}")
            
            # Excel 파일 찾기 (다양한 방법으로 시도)
            excel_files = []
            for pattern in ["*.xlsx", "*.xls"]:
                excel_files.extend(glob.glob(os.path.join(input_folder, pattern)))
            
            # 임시 파일 제외
            excel_files = [f for f in excel_files if not os.path.basename(f).startswith('~')]
            
            if not excel_files:
                self.add_log("ERROR: Excel 파일을 찾을 수 없습니다")
                return
            
            self.add_log(f"처리할 Excel 파일 {len(excel_files)}개 발견")
            
            success_count = 0
            
            for file_path in excel_files:
                if self.stop_requested:
                    break
                
                file_name = os.path.basename(file_path)
                self.add_log(f"파일 처리: {file_name}")
                
                try:
                    # 다양한 방법으로 파일 읽기 시도
                    all_sheets_data = self.read_excel_safely(file_path)
                    
                    if not all_sheets_data:
                        self.add_log(f"ERROR: 파일 읽기 실패 - {file_name}")
                        continue
                    
                    # 요약 시트 생성
                    summary_data = self.create_summary_sheet(all_sheets_data, file_name)
                    
                    if summary_data is not None:
                        # 기존 시트들과 함께 저장
                        all_sheets_data['요약'] = summary_data
                        
                        output_file_path = os.path.join(output_folder, file_name)
                        self.save_excel_safely(all_sheets_data, output_file_path)
                        
                        self.add_log(f"SUCCESS: {file_name} 처리 완료")
                        success_count += 1
                    else:
                        self.add_log(f"ERROR: 요약 생성 실패 - {file_name}")
                
                except Exception as e:
                    self.add_log(f"ERROR: {file_name} 처리 오류 - {str(e)[:100]}")
                    continue
            
            self.add_log(f"전체 처리 완료: {success_count}/{len(excel_files)} 성공")
            
        except Exception as e:
            self.add_log(f"ERROR: 전체 처리 오류 - {str(e)[:100]}")
        finally:
            self.root.after(0, self.finish_processing)
    
    def read_excel_safely(self, file_path):
        """다양한 방법으로 Excel 파일 안전하게 읽기"""
        all_sheets_data = {}
        
        # 방법 1: openpyxl 엔진
        try:
            excel_data = pd.ExcelFile(file_path, engine='openpyxl')
            for sheet_name in excel_data.sheet_names:
                try:
                    df = pd.read_excel(file_path, sheet_name=sheet_name, engine='openpyxl')
                    all_sheets_data[sheet_name] = df
                except Exception as e:
                    self.add_log(f"WARNING: 시트 '{sheet_name}' 읽기 실패 - {str(e)[:50]}")
                    continue
            
            if all_sheets_data:
                return all_sheets_data
        except Exception as e:
            self.add_log(f"WARNING: openpyxl 엔진 실패 - {str(e)[:50]}")
        
        # 방법 2: xlrd 엔진 (xls 파일용)
        try:
            excel_data = pd.ExcelFile(file_path, engine='xlrd')
            for sheet_name in excel_data.sheet_names:
                try:
                    df = pd.read_excel(file_path, sheet_name=sheet_name, engine='xlrd')
                    all_sheets_data[sheet_name] = df
                except:
                    continue
            
            if all_sheets_data:
                return all_sheets_data
        except:
            pass
        
        # 방법 3: 기본 엔진
        try:
            excel_data = pd.ExcelFile(file_path)
            for sheet_name in excel_data.sheet_names:
                try:
                    df = pd.read_excel(file_path, sheet_name=sheet_name)
                    all_sheets_data[sheet_name] = df
                except:
                    continue
            
            if all_sheets_data:
                return all_sheets_data
        except:
            pass
        
        self.add_log("ERROR: 모든 읽기 방법 실패")
        return {}
    
    def save_excel_safely(self, all_sheets_data, output_path):
        """Excel 파일 안전하게 저장"""
        try:
            # 방법 1: openpyxl
            with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
                for sheet_name, df in all_sheets_data.items():
                    df.to_excel(writer, sheet_name=sheet_name, index=False)
            return True
        except Exception as e:
            self.add_log(f"WARNING: openpyxl 저장 실패 - {str(e)[:50]}")
        
        try:
            # 방법 2: xlsxwriter
            with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
                for sheet_name, df in all_sheets_data.items():
                    df.to_excel(writer, sheet_name=sheet_name, index=False)
            return True
        except Exception as e:
            self.add_log(f"ERROR: xlsxwriter 저장도 실패 - {str(e)[:50]}")
            return False
    
    def create_summary_sheet(self, all_sheets_data, file_name):
        """기존 요약 시트에 AI 분석 결과 컬럼 추가"""
        try:
            # 기존 요약 시트 찾기
            original_summary = None
            if '요약' in all_sheets_data:
                original_summary = all_sheets_data['요약'].copy()
                self.add_log("기존 요약 시트 발견")
            else:
                self.add_log("WARNING: 기존 요약 시트가 없음")
                return None
            
            # URL목록 시트들 찾기
            url_sheets = []
            for sheet_name, df in all_sheets_data.items():
                if '예측_음식명' in df.columns:
                    url_sheets.append(sheet_name)
            
            if not url_sheets:
                self.add_log("WARNING: 예측 결과가 있는 시트를 찾을 수 없음")
                return original_summary
            
            self.add_log(f"분석할 시트: {url_sheets}")
            
            # 메뉴별 AI 분석 결과 수집
            ai_analysis_results = {}
            
            for sheet_name in url_sheets:
                df = all_sheets_data[sheet_name]
                
                # 메뉴 컬럼 찾기
                menu_col = None
                for col in df.columns:
                    if '메뉴' in str(col) or 'menu' in str(col).lower():
                        menu_col = col
                        break
                
                if not menu_col:
                    self.add_log(f"WARNING: '{sheet_name}' 시트에 메뉴 컬럼이 없음")
                    continue
                
                self.add_log(f"'{sheet_name}' 시트의 메뉴 컬럼: '{menu_col}'")
                
                # 메뉴별로 그룹화하여 분석
                for menu_name, group in df.groupby(menu_col):
                    if pd.isna(menu_name) or str(menu_name).strip() == '':
                        continue
                    
                    menu_name = str(menu_name).strip()
                    
                    # 오류가 아닌 유효한 예측들만 필터링
                    error_conditions = group['예측_음식명'].str.contains('오류|예측오류', na=False)
                    valid_predictions = group[group['예측_음식명'].notna() & ~error_conditions]
                    
                    if len(valid_predictions) == 0:
                        continue
                    
                    # 정확도 계산
                    correct_count = 0
                    wrong_predictions = {}
                    total_count = len(valid_predictions)
                    
                    for idx, row in valid_predictions.iterrows():
                        prediction = str(row['예측_음식명']).strip()
                        
                        # 메뉴명과 예측 결과 비교 (대소문자 무시, 공백 제거)
                        menu_clean = menu_name.lower().replace(' ', '').replace('_', '')
                        pred_clean = prediction.lower().replace(' ', '').replace('_', '')
                        
                        # 정확 예측 판단 (포함 관계로 유연하게)
                        if (menu_clean == pred_clean or 
                            menu_clean in pred_clean or 
                            pred_clean in menu_clean or
                            len(menu_clean) > 2 and menu_clean[:3] in pred_clean or
                            len(pred_clean) > 2 and pred_clean[:3] in menu_clean):
                            correct_count += 1
                        else:
                            # 오예측 기록
                            if prediction in wrong_predictions:
                                wrong_predictions[prediction] += 1
                            else:
                                wrong_predictions[prediction] = 1
                    
                    # 정확도 계산
                    accuracy = (correct_count / total_count * 100) if total_count > 0 else 0
                    
                    # 오예측 상위 3개
                    sorted_wrong = sorted(wrong_predictions.items(), key=lambda x: x[1], reverse=True)
                    
                    # 기존 결과와 병합 (여러 시트에서 같은 메뉴가 나올 수 있음)
                    if menu_name in ai_analysis_results:
                        # 기존 데이터와 합산
                        existing = ai_analysis_results[menu_name]
                        new_total = existing['AI_전체이미지수'] + total_count
                        new_correct = existing['AI_정확예측수'] + correct_count
                        new_accuracy = (new_correct / new_total * 100) if new_total > 0 else 0
                        
                        # 오예측 결과 병합
                        merged_wrong = existing['wrong_predictions'].copy()
                        for pred, count in wrong_predictions.items():
                            if pred in merged_wrong:
                                merged_wrong[pred] += count
                            else:
                                merged_wrong[pred] = count
                        
                        sorted_merged_wrong = sorted(merged_wrong.items(), key=lambda x: x[1], reverse=True)
                        
                        ai_analysis_results[menu_name] = {
                            'AI_전체이미지수': new_total,
                            'AI_정확예측수': new_correct,
                            'AI_정확률(%)': f"{new_accuracy:.1f}%",
                            'AI_오예측1위': sorted_merged_wrong[0][0] if len(sorted_merged_wrong) > 0 else "",
                            'AI_오예측1위_빈도': sorted_merged_wrong[0][1] if len(sorted_merged_wrong) > 0 else 0,
                            'AI_오예측2위': sorted_merged_wrong[1][0] if len(sorted_merged_wrong) > 1 else "",
                            'AI_오예측2위_빈도': sorted_merged_wrong[1][1] if len(sorted_merged_wrong) > 1 else 0,
                            'AI_오예측3위': sorted_merged_wrong[2][0] if len(sorted_merged_wrong) > 2 else "",
                            'AI_오예측3위_빈도': sorted_merged_wrong[2][1] if len(sorted_merged_wrong) > 2 else 0,
                            'wrong_predictions': merged_wrong
                        }
                    else:
                        # 새로운 메뉴 추가
                        ai_analysis_results[menu_name] = {
                            'AI_전체이미지수': total_count,
                            'AI_정확예측수': correct_count,
                            'AI_정확률(%)': f"{accuracy:.1f}%",
                            'AI_오예측1위': sorted_wrong[0][0] if len(sorted_wrong) > 0 else "",
                            'AI_오예측1위_빈도': sorted_wrong[0][1] if len(sorted_wrong) > 0 else 0,
                            'AI_오예측2위': sorted_wrong[1][0] if len(sorted_wrong) > 1 else "",
                            'AI_오예측2위_빈도': sorted_wrong[1][1] if len(sorted_wrong) > 1 else 0,
                            'AI_오예측3위': sorted_wrong[2][0] if len(sorted_wrong) > 2 else "",
                            'AI_오예측3위_빈도': sorted_wrong[2][1] if len(sorted_wrong) > 2 else 0,
                            'wrong_predictions': wrong_predictions
                        }
                    
                    self.add_log(f"'{menu_name}': {correct_count}/{total_count} ({accuracy:.1f}%)")
            
            if not ai_analysis_results:
                self.add_log("WARNING: AI 분석 결과가 없음")
                return original_summary
            
            # 기존 요약 시트에 AI 분석 컬럼들 추가
            ai_columns = ['AI_전체이미지수', 'AI_정확예측수', 'AI_정확률(%)', 
                         'AI_오예측1위', 'AI_오예측1위_빈도', 'AI_오예측2위', 
                         'AI_오예측2위_빈도', 'AI_오예측3위', 'AI_오예측3위_빈도']
            
            # '처리 내역' 열이 있으면 제거
            if '처리 내역' in original_summary.columns:
                original_summary = original_summary.drop('처리 내역', axis=1)
                self.add_log("'처리 내역' 열 제거됨")
            
            # 새 컬럼들을 빈 값으로 초기화
            for col in ai_columns:
                original_summary[col] = ""
            
            # 메뉴명 컬럼 찾기
            menu_column = None
            for col in original_summary.columns:
                if '메뉴' in str(col) or 'menu' in str(col).lower():
                    menu_column = col
                    break
            
            if not menu_column and len(original_summary.columns) > 0:
                menu_column = original_summary.columns[0]  # 첫 번째 컬럼을 메뉴명으로 가정
                self.add_log(f"메뉴 컬럼을 첫 번째 컬럼 '{menu_column}'로 가정")
            
            if menu_column is None:
                self.add_log("ERROR: 메뉴명 컬럼을 찾을 수 없음")
                return original_summary
            
            # 각 메뉴별로 AI 분석 결과 매칭하여 추가
            matched_count = 0
            total_rows = 0
            
            for idx, row in original_summary.iterrows():
                menu_name_in_summary = str(row[menu_column]).strip()
                
                # 빈 값이나 헤더가 아닌 경우만 처리
                if menu_name_in_summary and menu_name_in_summary != 'nan' and len(menu_name_in_summary) > 0:
                    total_rows += 1
                    
                    # ai_analysis_results에서 매칭되는 메뉴 찾기
                    found_results = None
                    matched_menu_key = None
                    
                    # 1. 정확히 일치하는 메뉴 찾기
                    if menu_name_in_summary in ai_analysis_results:
                        found_results = ai_analysis_results[menu_name_in_summary]
                        matched_menu_key = menu_name_in_summary
                    else:
                        # 2. 부분 일치 검색 (대소문자 무시, 공백/언더스코어 제거)
                        summary_clean = menu_name_in_summary.lower().replace(' ', '').replace('_', '')
                        for analysis_key in ai_analysis_results.keys():
                            analysis_clean = analysis_key.lower().replace(' ', '').replace('_', '')
                            if (summary_clean == analysis_clean or
                                summary_clean in analysis_clean or 
                                analysis_clean in summary_clean or
                                (len(summary_clean) > 2 and summary_clean[:3] == analysis_clean[:3])):
                                found_results = ai_analysis_results[analysis_key]
                                matched_menu_key = analysis_key
                                break
                    
                    if found_results:
                        matched_count += 1
                        # AI 분석 결과 추가 (wrong_predictions 제외)
                        for col in ai_columns:
                            if col in found_results:
                                original_summary.loc[idx, col] = found_results[col]
                        
                        self.add_log(f"매칭: '{menu_name_in_summary}' ↔ '{matched_menu_key}'")
                    else:
                        self.add_log(f"매칭 실패: '{menu_name_in_summary}'")
                        # 디버깅을 위해 사용 가능한 메뉴 목록 출력
                        available_menus = list(ai_analysis_results.keys())[:3]
                        self.add_log(f"  분석된 메뉴 샘플: {available_menus}")
            
            self.add_log(f"AI 분석 결과 추가 완료: {matched_count}/{total_rows} 메뉴 매칭됨")
            
            return original_summary
                
        except Exception as e:
            self.add_log(f"ERROR: 요약 시트 확장 오류 - {str(e)[:100]}")
            # 오류 시 기존 요약 시트라도 반환
            if '요약' in all_sheets_data:
                return all_sheets_data['요약']
            return None
    
    def finish_processing(self):
        self.is_running = False
        self.stop_requested = False
        self.start_button.config(state=tk.NORMAL)
        self.stop_button.config(state=tk.DISABLED)
        self.progress.stop()
        self.progress.config(value=0)
        self.status_var.set("완료")
    
    def run(self):
        self.root.mainloop()

def main():
    app = ExcelSummaryGenerator()
    app.run()

if __name__ == '__main__':
    main()